In [1]:
import numpy as np
import matplotlib.pyplot as plt 

import pandas as pd  
import seaborn as sns 
from sklearn import metrics
import scipy.io as sio
%matplotlib inline
from sklearn.model_selection import train_test_split
from sklearn import preprocessing
import warnings
warnings.filterwarnings('ignore')
import xarray as xr
from sklearn.preprocessing import PolynomialFeatures

In [2]:
## training data

week = '2'

file_path_obs_train = f'IMD_Data/Week{week}/IMD_pra_1989-2010.nc'
file_path_model_train = f'IMD_Data/Week{week}/GEFS_pra_1989-2010.nc'
data_obs_train = xr.open_dataset(file_path_obs_train)
data_model_train = xr.open_dataset(file_path_model_train)

## test data

file_path_obs_test = f'IMD_Data_New/Week{week}/IMD_pra_2022-2024.nc'
file_path_model_test = f'IMD_Data_New/Week{week}/GEFS_pra_2022-2024.nc'
data_obs_test = xr.open_dataset(file_path_obs_test)
data_model_test = xr.open_dataset(file_path_model_test)

In [3]:
data_model_test

<xarray.Dataset>
Dimensions:  (M: 31, Y: 32, S: 54, X: 31)
Coordinates:
  * M        (M) float32 0.0 1.0 2.0 3.0 4.0 5.0 ... 26.0 27.0 28.0 29.0 30.0
  * Y        (Y) float32 7.0 8.0 9.0 10.0 11.0 12.0 ... 34.0 35.0 36.0 37.0 38.0
  * S        (S) datetime64[ns] 2022-06-04 2022-06-11 ... 2024-09-21 2024-09-28
  * X        (X) float32 68.0 69.0 70.0 71.0 72.0 ... 94.0 95.0 96.0 97.0 98.0
Data variables:
    prcp     (S, M, Y, X) float64 ...

In [62]:
data_obs_test

<xarray.Dataset>
Dimensions:  (Y: 32, X: 31, T: 54)
Coordinates:
  * Y        (Y) float32 7.0 8.0 9.0 10.0 11.0 12.0 ... 34.0 35.0 36.0 37.0 38.0
  * X        (X) float32 68.0 69.0 70.0 71.0 72.0 ... 94.0 95.0 96.0 97.0 98.0
  * T        (T) datetime64[ns] 2022-06-14T12:00:00 ... 2024-10-08T12:00:00
Data variables:
    prcp     (T, Y, X) float32 nan nan nan nan nan nan ... nan nan nan nan nan

In [66]:
t_values = data_obs_test['T'].values  # Extract datetime values as numpy array

# Define target date in the same format
target_date = np.datetime64("2023-07-25T12:00:00.000000000")

# Find the index
index = np.where(t_values == target_date)[0]

print(index) 

[24]


In [64]:
t_values = data_model_test['S'].values  # Extract datetime values as numpy array

# Define target date in the same format
target_date = np.datetime64("2023-07-15T00:00:00.000000000")

# Find the index
index = np.where(t_values == target_date)[0]

print(index) 

[24]


In [4]:
train_obs_values = data_obs_train['prcp'].values
train_model_values = data_model_train.mean('M')['prcp'].values

In [5]:
test_obs_values = data_obs_test['prcp'].values
test_model_values = data_model_test.mean('M')['prcp'].values*86400

In [6]:
test_obs_values = test_obs_values[:50,:,:]

In [7]:
test_model_values = test_model_values[:50,:,:]

In [8]:
import json
import ast

# Read dictionary from a JSON file
with open(f'best_params_{week}_randomized.json', 'r') as json_file:
    params_str_keys = json.load(json_file)

# Convert string keys back to tuples
best_params_prop = {ast.literal_eval(k): v for k, v in params_str_keys.items()}

In [9]:
# # Read dictionary from a JSON file
# with open(f'best_params_extr_{week}.json', 'r') as json_file:
#     params_str_keys = json.load(json_file)

# # Convert string keys back to tuples
# best_params_prop1 = {ast.literal_eval(k): v for k, v in params_str_keys.items()}

In [10]:
# for i in best_params_prop1:
#     best_params_prop[i] = best_params_prop1[i]

In [11]:
def rps_clim(class_value):
    
    if len(class_value.shape)>1:
        cum_obs = np.cumsum(class_value, axis=1)
        cum_clim_probs = np.cumsum(np.ones((class_value.shape[0],3))*0.3333, axis=1).round(3)
        values = ((cum_clim_probs-cum_obs)**2)/(class_value.shape[1]-1)
    else:
        cum_obs = np.cumsum(class_value, axis=0)
        cum_clim_probs = np.cumsum(np.ones(3)*0.3333, axis=0).round(3)
        values = ((cum_clim_probs-cum_obs)**2)/(class_value.shape[0]-1)
    
    
    if len(class_value.shape)>1:
        return np.mean(np.sum(values, axis=1))
    else:
        return np.sum(values)
    
def rps_forecast(class_value, probs, grid_point=None):
    """
    grid_point: optional if only a single point is being fed as an input
    """
    if grid_point is not None:
        cum_obs = np.cumsum(class_value, axis=1)
        cum_fc_probs = np.cumsum(probs[grid_point[0], grid_point[1]], axis=1).round(3)
        values = ((cum_fc_probs-cum_obs)**2)/(class_value.shape[1]-1)
    else:
        cum_obs = np.cumsum(class_value, axis=1)
        cum_fc_probs = np.cumsum(probs, axis=1).round(3)
        values = ((cum_fc_probs-cum_obs)**2)/(class_value.shape[1]-1)
        
    
    if len(class_value.shape)>1:
        return np.mean(np.sum(values, axis=1))
    else:
        return np.sum(values)
    
def make_ohe(array):
    p_33 = np.quantile(array, q=1/3)
    p_67 = np.quantile(array, q=2/3)
    
    ohe = np.zeros((array.shape[0],3))
    for i in range(len(array)):
        if array[i]<=p_33:
            ohe[i][0] = 1
        elif p_33<array[i]<=p_67:
            ohe[i][1] = 1
        elif array[i]>p_67:
            ohe[i][2] = 1
    
    return ohe, p_33, p_67

In [12]:
def get_tercile_predictions_single_grid(observed_data_train, model_output_train, observed_data_test, model_output_test,
                                       mss, ne):
    '''
    Generate tercile probability predictions for observed test data based on a fitted quantile regression model for a single grid point.

    Parameters:
    observed_data_train (np.ndarray): 1D array containing the observed training data for a single grid point.
    model_output_train (np.ndarray): 1D array of model output for training for a single grid point.
    observed_data_test (np.ndarray): 1D array containing the observed testing data for a single grid point.
    model_output_test (np.ndarray): 1D array of model output for testing for a single grid point.

    Returns:
    np.ndarray: A 2D array of shape (number of test data points, 3), where each element contains the probabilities of falling into the lower, middle, and upper terciles.
    '''
    
    quantiles = np.linspace(0, 100, 40)
    
    ts_grid_ij_obs_train = observed_data_train
    ts_grid_ij_model_train = model_output_train
    ts_grid_ij_obs_test = observed_data_test
    ts_grid_ij_model_test = model_output_test

    if np.all(np.isnan(ts_grid_ij_obs_train)):
        return np.zeros((ts_grid_ij_obs_test.shape[0], 3))  # Return zeros if training data is all NaNs

    X_train = ts_grid_ij_model_train.reshape(-1, 1)
    y_train = ts_grid_ij_obs_train
    X_test = ts_grid_ij_model_test.reshape(-1, 1)
    y_test = ts_grid_ij_obs_test

    # Train the model
    rfqr = RandomForestQuantileRegressor(random_state=0, min_samples_split=mss, n_estimators=ne)
    rfqr.fit(X_train, y_train)

    # Predict quantiles for test data
    pred_test_quantiles = [rfqr.predict(X_test, quantile=q).reshape(-1, 1) for q in quantiles]
    q_preds_test = np.concatenate(pred_test_quantiles, axis=1)
    
    # Construct reference CDF
    reference_cdf = np.interp(np.linspace(np.min(ts_grid_ij_obs_train), np.max(ts_grid_ij_obs_train), y_test.shape[0]), 
                              np.quantile(y_train, np.linspace(0, 100, y_train.shape[0]) / 100), 
                              np.linspace(0, 100, y_train.shape[0]) / 100, left=0, right=1)

    # Calculate individual CDFs for each test point
    individual_cdfs = np.zeros((y_test.shape[0], y_test.shape[0]))
    for k in range(y_test.shape[0]):
        xp_k = q_preds_test[k, :]
        cdf_k = np.interp(np.linspace(np.min(xp_k), np.max(xp_k), y_test.shape[0]), 
                          xp_k, np.linspace(0, 100, 40) / 100, left=0, right=1)
        individual_cdfs[k, :] = cdf_k  # CDF for each point

    p1, p2 = 0.333, 0.667

    # Calculate the absolute difference from the specific value
    absolute_difference_from_value_1 = np.abs(reference_cdf - p1)
    absolute_difference_from_value_2 = np.abs(reference_cdf - p2)

    # Find the index of the minimum difference
    index_of_value_p33_ref = np.argmin(absolute_difference_from_value_1)
    index_of_value_p67_ref = np.argmin(absolute_difference_from_value_2)

    # Obtain the tercile predictions
    triplet_array = np.zeros((y_test.shape[0], 3))
    for k in range(y_test.shape[0]):
        # Take the correct individual CDF
        p_bn = individual_cdfs[k, :][index_of_value_p33_ref]
        p_n = individual_cdfs[k, :][index_of_value_p67_ref] - p_bn
        p_an = 1 - p_n - p_bn

        triplet_array[k, 0], triplet_array[k, 1], triplet_array[k, 2] = p_bn, p_n, p_an

    return triplet_array

In [17]:
from tqdm import tqdm
from skgarden import RandomForestQuantileRegressor

In [18]:
new_rpss_mat = np.full((32,31), np.nan)
terciles = np.full((32,31,50,3), np.nan)
ohe_mat = np.zeros((32,31,50,3))
rps_fc_mat = np.full((32,31),np.nan)
for i in tqdm(best_params_prop):

    if np.all(np.isnan(train_obs_values[:, i[0], i[1]])):
        print(f"Skipping {i[0], i[1]} as null...")
        continue
    ohe, _, _ = make_ohe(test_obs_values[:, i[0], i[1]])
    ohe_mat[i[0],i[1],:,:] = ohe
    rps_c = rps_clim(ohe)
    ms, n = best_params_prop[(i[0],i[1])]['mss'], best_params_prop[(i[0],i[1])]['ne']
    tp = get_tercile_predictions_single_grid(
        train_obs_values[:, i[0], i[1]], 
        train_model_values[:, i[0], i[1]], 
        test_obs_values[:, i[0], i[1]], 
        test_model_values[:, i[0], i[1]], ms, n
    )
    terciles[i[0],i[1],:,:] = tp
    rps_fc = rps_forecast(ohe, tp)
    rps_fc_mat[i[0],i[1]] = rps_fc
    new_rpss_mat[i[0], i[1]] = 1 - (rps_fc / rps_c)

  4%|█▌                                        | 12/323 [00:02<00:58,  5.36it/s]

Skipping (4, 12) as null...


  5%|██                                        | 16/323 [00:03<00:55,  5.54it/s]

Skipping (5, 12) as null...


 33%|█████████████▋                           | 108/323 [00:21<00:35,  6.09it/s]

Skipping (14, 19) as null...


... (5 outputs left out)

Skipping (21, 21) as null...


 91%|█████████████████████████████████████▎   | 294/323 [00:59<00:05,  5.07it/s]

Skipping (25, 11) as null...


100%|█████████████████████████████████████████| 323/323 [01:05<00:00,  4.97it/s]


In [21]:
np.save(f'tuned_rpss_week_{week}_2022_2024.npy', new_rpss_mat)

In [22]:
np.save(f'week_{week}_tuned_terciles_2022_2024.npy', terciles)

In [23]:
np.save(f'week_{week}_ohe_2022_2024.npy', terciles)

In [24]:
np.save(f'week_{week}_rps_fc_2022_2024.npy', rps_fc_mat)